### YELP Categories scraping
This notebook processes scraped business categories.

The YELP dataset contains many business categories with complex structure. Each category has its own subcategories, which, in turn, can have their own subcategories.

![Screenshot of structure of yelp categories](images/yelp_categories_screenshot.png)

All the categories are listed at [this site](https://business.yelp.com/resources/articles/yelp-category-list/?domain=local-business).

The html file with categories was obtained via:
```sh
wget -O yelp_categories.html --user-agent="Mozilla/5.0" https://business.yelp.com/resources/articles/yelp-category-list/?domain=local-business
```

Now, the categories have to be scraped.

In [1]:
from bs4 import BeautifulSoup
import xmltodict
from pprint import pp as pprint
import json

The categories are in `details` tags.

In [2]:
with open("original_data/yelp_json/yelp_categories.html") as html_doc:
    soup = BeautifulSoup(html_doc, 'html.parser')
details = soup.body.find_all(name="details")
details

[<details class="c-accordion-item js-accordion" data-item-image="" data-start-open="">
 <summary class="c-accordion-item__header js-accordion-title">
 <div class="c-accordion-item__title h4">
                     Active Life                </div>
 <div class="c-accordion-item__arrow"></div>
 </summary>
 <div class="c-accordion-item__content s-standard-content js-accordion-content">
 <p></p>
 <ul>
 <li>Airsoft</li>
 <li>Amateur Sports Teams</li>
 <li>Amusement Parks</li>
 <li>Aquariums</li>
 <li>Archery</li>
 <li>ATV Rentals/Tours</li>
 <li>Axe Throwing</li>
 <li>Badminton</li>
 <li>Baseball Fields</li>
 <li>Basketball Courts</li>
 <li>Bathing Area</li>
 <li>Batting Cages</li>
 <li>Beach Equipment Rentals</li>
 <li>Beach Volleyball</li>
 <li>Beaches</li>
 <li>Bicycle Paths</li>
 <li>Bike Parking</li>
 <li>Bike Rentals</li>
 <li>Boating</li>
 <li>Bobsledding</li>
 <li>Bocce Ball</li>
 <li>Bowling</li>
 <li>Bubble Soccer</li>
 <li>Bungee Jumping</li>
 <li>Canyoneering</li>
 <li>Carousels<

In [3]:
print(f"There are {len(details)} main categories.")

There are 21 main categories.


In [4]:
# functions used to process categories

def clean_dictionary(dictionary: dict, key = None) -> tuple[str, list]:
    cleaned = dictionary["ul"]["li"]
    if key == None:
        key = dictionary["#text"]
    return key, cleaned

def recur_convert_to_dict(dict_list: list, key=None, final_dict=dict()):
    if type(dict_list) == dict:
        key, clean_list = clean_dictionary(dictionary=dict_list, key=key)
    elif type(dict_list) == list:
        clean_list = dict_list
    elif type(dict_list) == str:
        clean_list = [dict_list]
    results = []
    for el in clean_list:
        if type(el) == str:
            results.append(el)
        elif type(el) == dict:
            new_key, el = clean_dictionary(dictionary=el)
            new_dict = recur_convert_to_dict(dict_list=el, key=new_key, final_dict=dict())
            results.append(new_dict)

    final_dict[key] = results
    return final_dict

In [5]:
# processing all the main categories and all their subcategories

for detail in details:
    main_category_name = detail.summary.div.contents[0].strip()
    dicti = xmltodict.parse(str(detail.ul))
    result_dict = recur_convert_to_dict(dict_list=dicti, key=main_category_name)


The categories are now in a simple dictionary. They are also easily parseable.

In [6]:
pprint(result_dict)

{'Active Life': ['Airsoft',
                 'Amateur Sports Teams',
                 'Amusement Parks',
                 'Aquariums',
                 'Archery',
                 'ATV Rentals/Tours',
                 'Axe Throwing',
                 'Badminton',
                 'Baseball Fields',
                 'Basketball Courts',
                 'Bathing Area',
                 'Batting Cages',
                 'Beach Equipment Rentals',
                 'Beach Volleyball',
                 'Beaches',
                 'Bicycle Paths',
                 'Bike Parking',
                 'Bike Rentals',
                 'Boating',
                 'Bobsledding',
                 'Bocce Ball',
                 'Bowling',
                 'Bubble Soccer',
                 'Bungee Jumping',
                 'Canyoneering',
                 'Carousels',
                 'Challenge Courses',
                 'Climbing',
                 'Cycling Classes',
                 'Dart Arenas',


In [ ]:
# saving the categories to a .json file
with open("custom_data/yelp_categories.json", "w") as categories_json:
    json.dump(result_dict, categories_json)